# Section 1 · Your First AI Agent (Groq edition)

In this notebook you build your first AI agent with the [Strands Agents SDK](https://strandsagents.com), powered by [Groq](https://console.groq.com) instead of Amazon Bedrock. Only the model setup (Steps 1–3) differs from `01_hello_agent.ipynb`; the agent code is identical.

You will:
1. Install the SDK
2. Pick a Groq model
3. Create an agent in one line
4. Give it your own tool with `@tool`
5. Give it a prebuilt tool
6. Shape its behaviour with a system prompt

**Before you start**
- A Groq API key from [console.groq.com/keys](https://console.groq.com/keys)
- Set it as `GROQ_API_KEY` in your environment or in a `.env` file (this folder or the repo root)

## Step 1: Install dependencies

In [1]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.


d:\work\AI Bootcamp\self_practice\myvenv\Scripts\python.exe: No module named pip


## Step 2: Choose the model

Every example in this course reads the model from one variable. Change it here (or set the `GROQ_MODEL_ID` environment variable) to try a different model.

- `openai/gpt-oss-120b` — default, strong reasoning and tool use
- `openai/gpt-oss-20b` — faster and cheaper
- `qwen/qwen3.8-27b` — an alternative open-weight model

See [console.groq.com/docs/models](https://console.groq.com/docs/models) for the current list.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

print(f"Model  : {MODEL_ID}")
print(f"API key: {'set' if os.getenv('GROQ_API_KEY') else 'MISSING - set GROQ_API_KEY'}")

Model  : openai/gpt-oss-120b
API key: set


## Step 3: Create your first agent

Groq exposes an OpenAI-compatible API, so Strands' `OpenAIModel` talks to it by pointing `base_url` at Groq. Swapping `BedrockModel` for this is the only change: everything after this cell is the same as the Bedrock notebook.

In [7]:
from strands import Agent
from strands.models.openai import OpenAIModel

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)

# An agent = a model + (optional) tools + (optional) system prompt
agent = Agent(model=model)

response = agent("Explain Amazon Bedrock AgentCore in three sentences.")

Amazon Bedrock AgentCore is the managed runtime service that powers “agents” built on Bedrock, handling the orchestration of large‑language‑model calls, memory management, and tool integration without requiring developers to write custom glue code. It provides a unified API for defining agent workflows—such as prompting, context‑aware reasoning, and invoking external functions or APIs—while automatically handling retries, throttling, and cost‑aware model selection. By abstracting these operational details, AgentCore lets users focus on the business logic of their conversational or autonomous applications, accelerating the creation of reliable, scalable AI agents.

The agent streams its answer as it generates it. The returned object holds the full result — the final text, token usage, and the stop reason:

In [ ]:
print("Stop reason:", response.stop_reason)
print("Tokens used:", response.metrics.accumulated_usage)

## Step 4: Give the agent a tool

A tool is just a Python function with:
- the `@tool` decorator
- type hints (so the model knows what arguments to send)
- a docstring (so the model knows *when* to use it)

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo

from strands import tool


@tool
def current_time(timezone: str = "Asia/Kolkata") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: IANA timezone name, for example "Asia/Kolkata" or "America/New_York".
    """
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y %I:%M %p %Z")


agent = Agent(model=model, tools=[current_time])
agent("What time is it right now in Bangalore and in New York?")


Tool #1: current_time

Tool #2: current_time
- **Bangalore (India – Asia/Kolkata):** Monday, 02 November 2026 01:31 PM IST  
- **New York (USA – America/New_York):** Monday, 02 November 2026 03:01 AM EST

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '- **Bangalore (India –\u202fAsia/Kolkata):**\u202fMonday,\u202f02\u202fNovember\u202f2026\u202f01:31\u202fPM\u202fIST  \n- **New\u202fYork (USA –\u202fAmerica/New_York):**\u202fMonday,\u202f02\u202fNovember\u202f2026\u202f03:01\u202fAM\u202fEST'}], 'metadata': {'usage': {'inputTokens': 252, 'outputTokens': 70, 'totalTokens': 322}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 466}}, 'tracking_id': 'eaa1322d-fd34-4f70-91cd-312f3be164f7'}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'current_time': ToolMetrics(tool={'toolUseId': 'fc_a4227f25-de23-40fb-8d8b-ff191f0c87b1', 'name': 'current_time', 'input': {'timezone': 'America/New_York'}}, call_count=2, success_count=2, error_count=0, total_time=0.09399999998277053)}, cycle_durations=[0.7198522090911865, 0.712080717086792, 0.6092538833618164], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='17eb546c-d06c

## Step 5: Use a prebuilt tool

`strands-agents-tools` ships ready-made tools. Here the agent gets a calculator so it does not have to do arithmetic "in its head".

In [13]:
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator, current_time])
agent("What is 18% GST on a laptop that costs Rs 84,999? Give me the final price too.")

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 84999*0.18          │                                                                            │
│  │ Result    │ 15299.82            │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 84999 + 15299.82    │                                                                            │
│  │ Result    │ 100298.82           │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**GST (18 %)** on a laptop priced at **Rs 84,999**:

- GST amount = Rs 84,999 × 0.18 = **Rs 15,299.82**

**Final price (including GST)**:

- Rs 84,999 + Rs 15,299.82 = **Rs 1,00,298.82**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '**GST (18\u202f%)** on a laptop priced at **Rs\u202f84,999**:\n\n- GST amount = Rs\u202f84,999\u202f×\u202f0.18 = **Rs\u202f15,299.82**\n\n**Final price (including GST)**:\n\n- Rs\u202f84,999\u202f+\u202fRs\u202f15,299.82 = **Rs\u202f1,00,298.82**.'}], 'metadata': {'usage': {'inputTokens': 1456, 'outputTokens': 86, 'totalTokens': 1542}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 599}}, 'tracking_id': '40f67a65-f5b8-4e75-bbbb-60bb4ce8a5ac'}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'fc_a4d74241-aeb4-449d-a234-770e8eeb1d81', 'name': 'calculator', 'input': {'expression': '84999 + 15299.82', 'mode': 'evaluate', 'precision': 2}}, call_count=2, success_count=2, error_count=0, total_time=0.15600000001722947)}, cycle_durations=[0.7864234447479248, 0.5700056552886963, 0.7597522735595703], agent_invocations=[AgentInvocation(cycles=[EventLoopCycle

## Step 6: Shape behaviour with a system prompt

The system prompt sets the agent's role, tone, and rules for every turn.

In [14]:
agent = Agent(
    model=model,
    system_prompt=(
        "You are a patient tutor for engineers who are new to AI agents. "
        "Explain ideas with one real-world analogy and keep answers under 120 words."
    ),
)
agent("What is the difference between an LLM and an AI agent?")

**Analogy:** Think of a *large language model* (LLM) as a **library** that contains millions of books and can instantly fetch or generate text based on what you ask. An *AI agent* is like a **librarian** who not only finds the right book but also decides which books to check out, talks to other libraries, and follows a goal (e.g., “plan a birthday party”).

**Difference:**  
- **LLM:** Purely a text‑generation engine; given a prompt it predicts the next words. It has no memory, goals, or ability to act on the outside world.  
- **AI Agent:** Wraps an LLM (or other tools) with a decision‑making loop, memory, and external actions (APIs, robots, databases). The agent uses the LLM to understand language, then executes steps to achieve a task.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '**Analogy:** Think of a *large language model* (LLM) as a **library** that contains millions of books and can instantly fetch or generate text based on what you ask. An *AI agent* is like a **librarian** who not only finds the right book but also decides which books to check out, talks to other libraries, and follows a goal (e.g., “plan a birthday party”).\n\n**Difference:**  \n- **LLM:** Purely a text‑generation engine; given a prompt it predicts the next words. It has no memory, goals, or ability to act on the outside world.  \n- **AI Agent:** Wraps an LLM (or other tools) with a decision‑making loop, memory, and external actions (APIs, robots, databases). The agent uses the LLM to understand language, then executes steps to achieve a task.'}], 'metadata': {'usage': {'inputTokens': 116, 'outputTokens': 228, 'totalTokens': 344}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 611}}, 'tracking_id':

## What you just did

- Built an agent with one line of code
- Added your own tool and a prebuilt tool
- Controlled behaviour with a system prompt
- Swapped the model provider from Bedrock to Groq without touching the agent code

Next, run the two Groq scripts in this folder to see the same idea as plain Python files, including one built with LangGraph:

```bash
python hello_agent_groq.py
python hello_langgraph_agent_groq.py
```